In [1]:
import gseapy as gp
import pandas as pd
import numpy as np

import os
import openpyxl
from sklearn.model_selection import train_test_split

import gc

In [2]:
!pwd

/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/Pilot_3_TCGA_Breast/notebooks/PASNet


In [3]:
os.chdir("/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/Pilot_3_TCGA_Breast")

In [4]:
### PATHWAY ENRICHMENT

In [9]:
# Load the Excel files
breast = pd.read_excel("results/TCGA_Breast/Breast_NetBID2_ms_tab.xlsx")

# Filtering based on conditions
pos = breast[(breast["adj.P.Val.Basal.Vs.LumA_DA"] < 0.05) & (breast["logFC.Basal.Vs.LumA_DA"] > 0) & (breast["Size"] > 30)]
neg = breast[(breast["adj.P.Val.Basal.Vs.LumA_DA"] < 0.05) & (breast["logFC.Basal.Vs.LumA_DA"] < 0) & (breast["Size"] > 30)]

# Row-bind (combine) the results
ms_tab = pd.concat([pos, neg], axis=0, ignore_index=True)

# Get unique values from the 'originalID' column
ms_tab = list(ms_tab['originalID'].unique())
ms_tab.sort()

# Display the result
len(ms_tab)

7538

In [6]:
# Load the activity matrix from the CSV file
activity = pd.read_csv("/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/data/TCGA_Breast/brca_epithelial_activity.csv")

# Clean the feature names by removing suffixes (_TF, _SIG, _LNC) to standardize identifiers
activity['Unnamed: 0'] = activity['Unnamed: 0'].str.replace('_TF', '').str.replace('_SIG', '').str.replace('_LNC', '')

# Remove duplicate entries for the same feature, set the names as the index, 
# and transpose the matrix so that samples are rows and features are columns
activity = activity.drop_duplicates(subset=['Unnamed: 0']).set_index('Unnamed: 0').T

# Extract the processed feature names as a column index
ms_tab_sc = activity.columns

# Display the total number of unique features in the final dataset
print(len(ms_tab_sc))

10723


In [7]:
# Χρήση set intersection για εύρεση κοινών στοιχείων
ms_tab = sorted(list(set(ms_tab_sc) & set(ms_tab)))

print(len(ms_tab))

4507


In [8]:
# For Gene Ontology
# Run ORA with a custom GMT file
enr = gp.enrichr(
    gene_list=ms_tab, 
    gene_sets="./data/EnrichR/merged_GO_Biological_Process_2021.gmt",  # Path to your custom GMT file
    organism="human", 
    outdir="./data/EnrichR/both/merged_GO_Biological_Process_2021_res",
    cutoff=0.1
)

results_go = enr.results

results_go_filt = results_go[(results_go['P-value'] < 0.05)]
len(results_go_filt)

562

In [9]:
results_go_filt = results_go[(results_go['Adjusted P-value'] < 0.05)]
len(results_go_filt)

92

In [10]:
results_go_filt['Genes'] = results_go_filt['Genes'].str.split(';')

results_go_filt_n = results_go_filt.explode("Genes").pivot_table(index="Term", columns="Genes", aggfunc="size", fill_value=0).reset_index()
results_go_filt_n = results_go_filt_n.set_index('Term')

results_go_filt_n.to_excel("./PASNet/Input/GO/pt_fixed.xlsx")

/tmp/ipykernel_4343/611029904.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_go_filt['Genes'] = results_go_filt['Genes'].str.split(';')


In [19]:
# Garbage collection
gc.collect()

working_dir = "./"

# Load activity data
activity = pd.read_csv(working_dir + "data/TCGA_Breast/activity_matrix.csv")
activity['Unnamed: 0'] = activity['Unnamed: 0'].str.replace('_TF', '').str.replace('_SIG', '').str.replace('_NC', '')
activity = activity.drop_duplicates(subset=['Unnamed: 0']).set_index('Unnamed: 0').T

# Load metadata
metadata = pd.read_csv(working_dir + "data/TCGA_Breast/metadata.csv", index_col=0)
metadata = metadata[['paper_BRCA_Subtype_PAM50']]
metadata['paper_BRCA_Subtype_PAM50'] = metadata['paper_BRCA_Subtype_PAM50'].replace({'Basal': '1', 'LumA': '0'})

# Load pathway data
pt_go = pd.read_excel(working_dir + "Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/pt_fixed.xlsx", index_col=0)

# Subset activity data
activity_go = activity.loc[:, activity.columns.isin(pt_go.columns)]

# Merge with metadata
activity_go = activity_go.merge(metadata, left_index=True, right_index=True)

# Split into train and test sets
train_go, validation_go = train_test_split(activity_go, test_size=0.2, random_state=123, stratify=activity_go['paper_BRCA_Subtype_PAM50'])

# Save to Excel
train_go.to_excel(working_dir + "Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/Training.xlsx", index=True)
validation_go.to_excel(working_dir + "Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/Validation_grinding.xlsx", index=True)

In [20]:
pt_go_filt = pt_go.loc[:, pt_go.columns.isin(activity_go.columns)]

In [21]:
pt_go_filt.to_excel("/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/pt_fixed.xlsx")

In [22]:
## Filter scRNAseq activity table

In [23]:
working_dir = "/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/"

# Load activity data (epithelial)
activity = pd.read_csv(working_dir + "data/TCGA_Breast/brca_epithelial_activity_matrix_one_network.csv")
activity['Unnamed: 0'] = activity['Unnamed: 0'].str.replace('_TF', '').str.replace('_SIG', '').str.replace('_LNC', '')
activity = activity.drop_duplicates(subset=['Unnamed: 0']).set_index('Unnamed: 0').T

# Load metadata (epithelial)
metadata = pd.read_csv(working_dir + "data/TCGA_Breast/brca_epithelial_metadata_one_network.csv", index_col=0)
metadata = metadata[['celltype_minor']]
metadata['celltype_minor'] = metadata['celltype_minor'].replace({'Basal': '1', 'LumA': '0'}) # Adjusted to catch both Luminal types if present

In [24]:
# Load pathway data
pt_go = pd.read_excel("/mnt/c/Users/vvasileiou/Desktop/test_lncAPNet/Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/pt_fixed.xlsx", index_col=0)

# Subset activity data
activity_go = activity.loc[:, activity.columns.isin(pt_go.columns)]

In [25]:
# Merge with metadata
activity_go = activity_go.merge(metadata, left_index=True, right_index=True)

In [26]:
activity_go = activity_go.rename(columns={'celltype_minor': 'paper_BRCA_Subtype_PAM50'})
# Reorder activity_go to match the sequence of train_go
activity_go = activity_go[train_go.columns]
activity_go.to_excel(working_dir + "Pilot_3_TCGA_Breast/PASNet/Input_one_network/GO/Testing.xlsx", index=True)

In [27]:
if list(train_go.columns) == list(activity_go.columns):
    print("✅ Orders match perfectly!")
else:
    print("❌ Orders are still different.")

✅ Orders match perfectly!
